# Entrenamiento del Modelo de Detección de Fraude

Este cuaderno detalla el proceso de preprocesamiento, entrenamiento, evaluación y exportación del modelo de detección de fraude.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import os

# Configuración
data_path = '../data/financial_fraud_detection_dataset.csv'
model_path = '../model/fraud_model.joblib'

## 1. Carga y Preparación de Datos

In [ ]:
df = pd.read_csv(data_path)

# Selección de características relevantes
features = ['amount', 'transaction_type', 'merchant_category', 'device_used', 
            'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'payment_channel']
target = 'is_fraud'

X = df[features]
y = df[target].astype(int) # True/False a 1/0

# Dividir datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Entrenamiento: {X_train.shape}, Prueba: {X_test.shape}")

## 2. Creación del Pipeline de Preprocesamiento y Modelo

In [ ]:
# Definir preprocesamiento para variables numéricas y categóricas
numeric_features = ['amount', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score']
categorical_features = ['transaction_type', 'merchant_category', 'device_used', 'payment_channel']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Crear el pipeline completo con el clasificador
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'))
])

## 3. Entrenamiento

In [ ]:
model_pipeline.fit(X_train, y_train)

## 4. Evaluación

In [ ]:
y_pred = model_pipeline.predict(X_test)
y_prob = model_pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Score: {roc_auc_score(y_test, y_prob):.4f}")

## 5. Guardar el Modelo

In [ ]:
# Crear carpeta si no existe (aunque ya debería estar creada)
os.makedirs(os.path.dirname(model_path), exist_ok=True)

joblib.dump(model_pipeline, model_path)
print(f"Modelo guardado exitosamente en: {model_path}")